# Verde Bowl — Input Guardrail LoRA Fine-Tune (Phase 6)

Trains a small model (Qwen2.5-1.5B-Instruct) to label each message as one of:
`SAFE / OFF_SCOPE / JAILBREAK / INJECTION / POLICY_LEAK / PII / ENCODED`.

**How to run:**
1. `Runtime → Change runtime type → T4 GPU`.
2. `Runtime → Run all`.
3. When the **upload** cell runs, pick the 3 files from your local `finetune/data/`: `train.jsonl`, `val.jsonl`, `test.jsonl`.
4. At the end it prints held-out accuracy + per-class recall and downloads `verde_adapter.zip`.
5. Unzip that into your repo at `finetune/adapter/` — the v1 guardrail picks it up automatically.

~10–15 min on a free T4.

In [ ]:
!nvidia-smi -L    # confirm a GPU is attached (else: Runtime > Change runtime type > T4 GPU)

In [ ]:
!pip install -q -U "transformers>=4.44" "peft>=0.12" "datasets>=2.20" "accelerate>=0.33" bitsandbytes

In [ ]:
# Upload the 3 dataset files from your local finetune/data/ folder
from google.colab import files
up = files.upload()   # select train.jsonl, val.jsonl, test.jsonl
assert all(f in up for f in ['train.jsonl','val.jsonl','test.jsonl']), 'Please upload all three .jsonl files'
print('uploaded:', list(up.keys()))

In [ ]:
# ---- Train (LoRA, 4-bit) ----
import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, Trainer, DataCollatorForSeq2Seq)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE = 'Qwen/Qwen2.5-1.5B-Instruct'   # swap to 'Qwen/Qwen3-1.7B' if you prefer
MAXLEN = 512

tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def build(ex):
    msgs = [{'role':'system','content':ex['instruction']}, {'role':'user','content':ex['input']}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    full = prompt + ex['output'] + tok.eos_token
    p_ids = tok(prompt, add_special_tokens=False)['input_ids']
    f_ids = tok(full, add_special_tokens=False)['input_ids'][:MAXLEN]
    labels = ([-100]*len(p_ids) + f_ids[len(p_ids):])[:MAXLEN]
    return {'input_ids': f_ids, 'attention_mask':[1]*len(f_ids), 'labels': labels}

ds = load_dataset('json', data_files={'train':'train.jsonl','val':'val.jsonl'})
train = ds['train'].map(build, remove_columns=ds['train'].column_names)
val   = ds['val'].map(build,   remove_columns=ds['val'].column_names)

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto')
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']))
model.print_trainable_parameters()

args = TrainingArguments(output_dir='runs', num_train_epochs=3,
    per_device_train_batch_size=8, gradient_accumulation_steps=2, learning_rate=2e-4,
    warmup_ratio=0.05, logging_steps=10, eval_strategy='epoch', save_strategy='no',
    fp16=True, report_to='none')
trainer = Trainer(model=model, args=args, train_dataset=train, eval_dataset=val,
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
trainer.train()
model.save_pretrained('adapter'); tok.save_pretrained('adapter')
print('saved adapter/')

In [ ]:
# ---- Evaluate on the held-out ORIGINAL test split ----
import json, collections, torch
CLASSES = ['SAFE','OFF_SCOPE','JAILBREAK','INJECTION','POLICY_LEAK','PII','ENCODED']
test = [json.loads(l) for l in open('test.jsonl', encoding='utf-8')]
model.eval()
correct = 0; per = collections.defaultdict(lambda:[0,0]); conf = collections.Counter()
for ex in test:
    msgs = [{'role':'system','content':ex['instruction']}, {'role':'user','content':ex['input']}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=6, do_sample=False, pad_token_id=tok.pad_token_id)
    gen = tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).upper()
    pred = next((c for c in CLASSES if c in gen), 'SAFE'); gold = ex['output']
    per[gold][1] += 1
    if pred == gold: correct += 1; per[gold][0] += 1
    else: conf[f'{gold}->{pred}'] += 1
print(f'\naccuracy {correct}/{len(test)} = {100*correct/len(test):.1f}%')
for c in CLASSES:
    r,t = per[c]
    if t: print(f'  recall {c:12s} {r}/{t} = {100*r/t:.0f}%')
print('  confusions:', dict(conf))

In [ ]:
# ---- Download the adapter (unzip into repo finetune/adapter/) ----
import shutil
from google.colab import files
shutil.make_archive('verde_adapter', 'zip', 'adapter')
files.download('verde_adapter.zip')